# Semantic Similarity: SBERT Embeddings & t-SNE

**Ship of Theseus: Computational Forensics**

This notebook computes SBERT cosine similarity between T0 and T1-T3 texts, and generates t-SNE identity trajectory visualizations from SBERT embeddings.

**Designed for Google Colab** (GPU recommended for SBERT encoding).

---

In [ ]:
# Colab setup: install dependencies
# Uncomment the next line if running on Google Colab
# !pip install sentence-transformers pandas numpy matplotlib seaborn scikit-learn

import sys
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler

sns.set_theme(style='whitegrid', font_scale=1.1)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

# Detect environment
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = Path('/content/drive/MyDrive/ShipOfTheseus')
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    print(f'Running on Colab. Output: {OUTPUT_DIR}')
else:
    OUTPUT_DIR = Path.cwd()
    print(f'Running locally. Output: {OUTPUT_DIR}')


## 1. Load Data

Load the multitier results from the cached pickle file.
If running on Colab, upload `multitier_results.pkl` to your Drive first.

In [ ]:
# Load data — auto-detects Colab vs local
if IN_COLAB:
    DATA_PATH = OUTPUT_DIR / 'multitier_results.pkl'
    if not DATA_PATH.exists():
        print(f'ERROR: Upload multitier_results.pkl to {OUTPUT_DIR}')
        print('Go to Google Drive > MyDrive > ShipOfTheseus/ and upload the file.')
        raise FileNotFoundError(f'{DATA_PATH} not found')
else:
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
    DATA_PATH = PROJECT_ROOT / 'data' / 'processed' / 'multitier_results.pkl'

with open(DATA_PATH, 'rb') as f:
    multitier = pickle.load(f)

print(f'Loaded {len(multitier)} paraphrasers')
for k, df in multitier.items():
    print(f'  {k}: {len(df)} articles')


## 2. SBERT Encoding

Encode texts at each tier using `all-MiniLM-L6-v2` (384-dim). We use Dipper (3,010 articles) as the primary paraphraser for balanced analysis.

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
print(f'SBERT model loaded on {device}')

# Use Dipper for balanced source distribution
df = multitier['dipper'].copy()
print(f'Articles: {len(df)}')

# Encode all tiers
sbert_embeddings = {}
for tier in ['T0', 'T1', 'T2', 'T3']:
    text_col = f'text_{tier}'
    mask = df[text_col].notna()
    texts = df.loc[mask, text_col].tolist()
    print(f'Encoding {tier}: {len(texts)} texts...')
    embs = model.encode(texts, batch_size=64, show_progress_bar=True, convert_to_numpy=True)
    sbert_embeddings[tier] = (embs, mask)
    print(f'  Shape: {embs.shape}')

# Cache embeddings so we don't have to re-encode
emb_cache = OUTPUT_DIR / 'sbert_embeddings.pkl'
import pickle as _pkl
with open(emb_cache, 'wb') as _f:
    _pkl.dump(sbert_embeddings, _f)
print(f'Cached embeddings to {emb_cache}')


## 3. SBERT Cosine Similarity (T0 vs Tn)

In [ ]:
print('SBERT Cosine Similarity (T0 vs Tn):\n')
sbert_cosines = {}
emb_t0, mask_t0 = sbert_embeddings['T0']

for tier in ['T1', 'T2', 'T3']:
    emb_tn, mask_tn = sbert_embeddings[tier]
    # Align: use rows present in both T0 and Tn
    common_mask = mask_t0 & mask_tn
    idx_t0 = mask_t0[common_mask].index
    idx_tn = mask_tn[common_mask].index
    
    # Reindex embeddings to common set
    e0 = emb_t0[:common_mask.sum()]
    en = emb_tn[:common_mask.sum()]
    
    # Cosine similarity (row-wise)
    cos = np.sum(e0 * en, axis=1) / (np.linalg.norm(e0, axis=1) * np.linalg.norm(en, axis=1))
    sbert_cosines[tier] = cos
    print(f'{tier}: mean={cos.mean():.4f} +/- {cos.std():.4f} (n={len(cos)})')


## 4. Comparison with BERTScore

SBERT cosine should correlate with BERTScore. Both measure semantic preservation, but SBERT uses sentence-level embeddings while BERTScore uses token-level matching.

In [ ]:
# Compare SBERT vs BERTScore (from baselines)
# On Colab: upload similarity_baselines.csv to Drive, or paste the values below
if IN_COLAB:
    baselines_path = OUTPUT_DIR / 'similarity_baselines.csv'
else:
    baselines_path = PROJECT_ROOT / 'experiments' / 'baseline_results' / 'similarity_baselines.csv' 

# Fallback: hardcoded BERTScore values for Dipper (from Phase I results)
DIPPER_BERTSCORE = {'T1': 0.9045, 'T2': 0.8883, 'T3': 0.8770}

if baselines_path.exists():
    baselines = pd.read_csv(baselines_path)
    dipper_bl = baselines[baselines['Paraphraser_Key'] == 'dipper']
    bert_vals = {}
    for tier in ['T1', 'T2', 'T3']:
        row = dipper_bl[dipper_bl['Tier'] == tier]
        bert_vals[tier] = row['BERTScore'].iloc[0] if len(row) > 0 else DIPPER_BERTSCORE[tier]
else:
    print('Baselines CSV not found — using hardcoded BERTScore values from Phase I')
    bert_vals = DIPPER_BERTSCORE

print('Dipper: SBERT vs BERTScore comparison
')
print(f'{"Tier":<6s} {"SBERT Cosine":>15s} {"BERTScore":>15s}')
print('-' * 40)
for tier in ['T1', 'T2', 'T3']:
    sbert_val = sbert_cosines[tier].mean()
    print(f'{tier:<6s} {sbert_val:>15.4f} {bert_vals[tier]:>15.4f}')

print('
Both metrics confirm semantic preservation across tiers.')


## 5. t-SNE on SBERT Embeddings

Visualize how documents move through 384-dim semantic space across tiers.

In [ ]:
# Stack all embeddings with tier labels
all_embs = []
all_tier_labels = []
all_source_labels = []

for tier in ['T0', 'T1', 'T2', 'T3']:
    embs, mask = sbert_embeddings[tier]
    sources = df.loc[mask, 'source'].values
    all_embs.append(embs)
    all_tier_labels.extend([tier] * len(embs))
    all_source_labels.extend(sources)

X_all = np.vstack(all_embs)
tiers_arr = np.array(all_tier_labels)
sources_arr = np.array(all_source_labels)
binary_src = np.array(['Human' if s == 'Human' else 'LLM' for s in sources_arr])

print(f'Running t-SNE on {X_all.shape[0]} x {X_all.shape[1]} SBERT embeddings...')
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=1000,
            learning_rate='auto', init='pca')
X_2d = tsne.fit_transform(X_all)
print(f'Done. Shape: {X_2d.shape}')


In [ ]:
# Plot t-SNE colored by tier
tier_colors = {'T0': '#2ecc71', 'T1': '#f39c12', 'T2': '#e74c3c', 'T3': '#9b59b6'}

fig, axes = plt.subplots(1, 2, figsize=(16, 7))

ax = axes[0]
for tier in ['T3', 'T2', 'T1', 'T0']:
    mask = tiers_arr == tier
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=tier_colors[tier],
               label=tier, alpha=0.3, s=8, edgecolors='none')
ax.set_title('SBERT Trajectory: By Tier', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=10, markerscale=3)

ax = axes[1]
src_colors = {'Human': '#2ecc71', 'LLM': '#e74c3c'}
for tier, marker, alpha in [('T3', 'x', 0.2), ('T0', 'o', 0.5)]:
    for src in ['LLM', 'Human']:
        mask = (tiers_arr == tier) & (binary_src == src)
        ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=src_colors[src],
                   marker=marker, label=f'{src} ({tier})', alpha=alpha, s=12)
ax.set_title('SBERT Trajectory: Human vs LLM (T0 vs T3)', fontsize=13, fontweight='bold')
ax.set_xlabel('t-SNE 1')
ax.set_ylabel('t-SNE 2')
ax.legend(fontsize=9, markerscale=2, loc='upper right')

fig.suptitle('Identity Trajectory in SBERT Embedding Space (Dipper)',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'sbert_identity_trajectory.png', dpi=300, bbox_inches='tight')
plt.show()
print(f'Saved to {OUTPUT_DIR / "sbert_identity_trajectory.png"}')


## 6. Save Results

Cache embeddings and cosine scores for local use.

In [ ]:
# Save SBERT cosine results
sbert_results = {
    'cosines': {t: c.tolist() for t, c in sbert_cosines.items()},
    'means': {t: float(c.mean()) for t, c in sbert_cosines.items()},
}

output_path = OUTPUT_DIR / 'sbert_results.pkl'
with open(output_path, 'wb') as f:
    pickle.dump(sbert_results, f)
print(f'Saved to {output_path}')

print('
SBERT Cosine Summary:')
for t, m in sbert_results['means'].items():
    print(f'  {t}: {m:.4f}')


## 7. Download Results for Local Use

After running this notebook on Colab, download the following files from **Google Drive > MyDrive > ShipOfTheseus/** and place them in your local repo:

| Colab file | Local destination |
|---|---|
|  |  |
|  |  |
|  |  |

Or run the cell below to download directly from Colab:

In [ ]:
if IN_COLAB:
    from google.colab import files
    for f in ["sbert_results.pkl", "sbert_embeddings.pkl", "sbert_identity_trajectory.png"]:
        path = OUTPUT_DIR / f
        if path.exists():
            files.download(str(path))
            print(f"Downloaded {f}")
        else:
            print(f"Not found: {path}")
else:
    print("Not on Colab. Files are already saved locally.")


---
## Summary

SBERT cosine similarity confirms the semantic preservation pattern observed with BERTScore. Both metrics show that meaning remains largely intact through T3, even as lexical surface and entity content erode. The t-SNE visualization in SBERT space provides a complementary view to the combined-feature-space trajectory, showing how documents cluster by tier in semantic embedding space.